**This notebook is for live weather data stream.**

In [0]:
import requests
import pandas as pd

dbutils.widgets.text('LAT','30.4745',"Latitude")
dbutils.widgets.text('LON','78.1551',"Longitude")

def fetch_daily_mussoorie_weather():
    # Mussoorie Coordinates
    LAT = dbutils.widgets.get('LAT')
    LON = dbutils.widgets.get('LON')
    
    # The exact parameters from your PySpark schema
    METRICS = "temperature_2m,relative_humidity_2m,cloud_cover_mid,cloud_cover_high,cloud_cover_low,rain,snowfall,precipitation,dew_point_2m,wind_speed_100m,wind_speed_10m,wind_direction_10m"
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": LAT,
        "longitude": LON,
        "hourly": METRICS,
        "past_days": 7,
        "forecast_days": 16
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Load directly into a Pandas DataFrame using the JSON dictionary
    df_live = pd.DataFrame(data["hourly"])
    
    return df_live

df_new_data = fetch_daily_mussoorie_weather()
df_new_data['time'] = pd.to_datetime(df_new_data['time'])

# display(df_new_data)

In [0]:
%run ./historic_bronze

In [0]:
# MAGIC %run ./weather_data_processing

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# 1. Convert the Pandas DataFrame to Spark, enforcing your exact schema
spark_live_raw = spark.createDataFrame(df_new_data, schema=schema)

# 2. Add the exact raw metadata columns present in bronze_df to maintain uniformity
spark_live_bronze = spark_live_raw \
    .withColumn('read_timestamp', F.current_timestamp()) 
    # .withColumn('file_name', F.lit('API_LIVE_STREAM')) \
    # .withColumn('file_size', F.lit(0).cast('long'))



In [0]:
spark_live_bronze.describe()

In [0]:
# 3. Perform an in-memory Delta Merge into your existing bronze Delta table

bronze_delta_table = DeltaTable.forName(spark, "weather.bronze.bronze_weather")

(bronze_delta_table.alias("target")
 .merge(
     spark_live_bronze.alias("source"),
     "target.time = source.time"
 )
 .whenMatchedUpdate(set={
     "temperature_2m": "source.temperature_2m",
     "relative_humidity_2m": "source.relative_humidity_2m",
     "cloud_cover_mid": "source.cloud_cover_mid",
     "cloud_cover_high": "source.cloud_cover_high",
     "cloud_cover_low": "source.cloud_cover_low",
     "rain": "source.rain",
     "snowfall": "source.snowfall",
     "precipitation": "source.precipitation",
     "dew_point_2m": "source.dew_point_2m",
     "wind_speed_100m": "source.wind_speed_100m",
     "wind_speed_10m": "source.wind_speed_10m",
     "wind_direction_10m": "source.wind_direction_10m",
     "read_timestamp": "source.read_timestamp"
 })
 .whenNotMatchedInsertAll()
 .execute()
)

bronze_df = spark.table("weather.bronze.bronze_weather")

In [0]:
# Find duplicate rows in bronze_df including metadata columns
metadata_cols = ['read_timestamp']  # Add other metadata columns if present
all_cols = bronze_df.columns

duplicates_df = bronze_df.groupBy(all_cols) \
    .count() \
    .filter("count > 1")

display(duplicates_df)

In [0]:
# Save the bronze DataFrame as a permanent Delta table in the metastore
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather.bronze.bronze_weather")


In [0]:
from pyspark.sql.functions import col, sum

null_counts = bronze_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in bronze_df.columns])
display(null_counts)